# Práctica 3: A\*, Hill Climbing y Optimización de Redes

Fabian Prado - 23427 | CC3074 Modelación y Simulación, UVG | 2026-10-06

**Pregunta:** ¿Qué solución encuentran Hill Climbing, A\*, Edmonds-Karp y el flujo de costo mínimo en sus tres problemas, y cómo se comprueba si es la mejor posible?

El laboratorio tiene tres problemas, cada uno con su algoritmo:

1. **Hill Climbing** para el agente viajero: recorrer las ciudades 2 a 7 saliendo y volviendo a la ciudad 1 con la menor distancia.
2. **A\*** para la ruta más corta de O a T en la red del guardabosques.
3. **Flujo Máximo (Edmonds-Karp) y Costo Mínimo** para enviar suministros de S a T.

Todos los algoritmos están implementados desde cero, solo con la librería estándar de Python (`heapq`, `collections`, `random`, `itertools`, `math`) y `matplotlib` para las gráficas, como pide el enunciado.

## Mapa del notebook

1. Imports y configuración
2. Obtención de datos
3. Preparación de datos
4. Variables
5. Modelos y métricas
6. Gráficas
7. Ejecución
8. Conclusiones

Las secciones 2 a 6 solo definen funciones, separadas por parte. La sección 7 las ejecuta en orden y cada resultado lleva su interpretación.

## Dónde está cada actividad

| Parte | Actividades del enunciado | Paso en la sección 7 |
|---|---|---|
| 1. Hill Climbing | 1. `distancia_tour` y tour inicial | 7.1 |
| | 2. `vecinos` por inversión | 7.2 |
| | 3 y 4. Hill Climbing de máximo descenso con registro | 7.3 |
| | 5. 100 reinicios aleatorios | 7.4 |
| | Extra: todos los tours válidos por fuerza bruta | 7.5 |
| | 6. Gráfica del mejor recorrido | 7.6 |
| | Preguntas de análisis | después de 7.6 |
| 2. A\* | 7. `heuristica` euclidiana | 7.7 |
| | 8, 9 y 10. A\* con `heapq`, registro, ruta y costo | 7.8 |
| | 11. A\* con h(n) = 0 | 7.9 |
| | 12. Gráfica de la red | 7.10 |
| | Preguntas de análisis | después de 7.10 |
| 3A. Flujo Máximo | 13. Red residual | 7.11 |
| | 14, 15 y 16. Edmonds-Karp con registro y flujo por tramo | 7.12 |
| | 17. Verificación de capacidad y conservación | 7.13 |
| | Extra: corte mínimo | 7.14 |
| 3B. Costo Mínimo | 18. Capacidad, costo y flujo por arista | 7.15 |
| | 19 a 23. Costo mínimo con registro | 7.16 |
| | 24. Reporte final | 7.17 |
| | Extra: costo de la distribución de Edmonds-Karp | 7.18 |
| | Preguntas de análisis | después de 7.18 |
| Integración | Tabla comparativa y conclusión | 8 |

## 1. Imports y configuración

Todas las librerías, la semilla y la paleta en una sola celda. Solo se usan las librerías que permite el enunciado: nada de `numpy`, `pandas` ni `networkx`, así que las tablas se imprimen con texto. `SEED` fija los reinicios aleatorios de Hill Climbing para que cualquier corrida dé los mismos números.

In [1]:
import heapq
import itertools
import math
import random
from collections import deque
from pathlib import Path

import matplotlib.pyplot as plt

SEED = 42
ROOT = Path.cwd()
FIG_DIR = ROOT / "figuras"
FIG_DIR.mkdir(exist_ok=True)

# Forest, leaf, ochre, lake, clay: greens first, earth and water tones for more series.
PALETTE = ["#0E4D45", "#3F8A26", "#B07D2B", "#3A7CA5", "#A4493D"]
plt.rcParams.update({
    "axes.prop_cycle": plt.cycler(color=PALETTE),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "figure.dpi": 110,
    "figure.figsize": (16, 9),
    "figure.facecolor": "white",
    "legend.frameon": True,
})

## 2. Obtención de datos

TODO: de dónde vienen los datos, cómo se obtienen y qué devuelve cada función.

In [2]:
# TODO: definitions only, no calls here

## 3. Preparación de datos

TODO: qué limpieza o transformación se hace y por qué.

In [3]:
# TODO: definitions only, no calls here

## 4. Variables

TODO: qué variables se construyen y por qué ayudan a responder la pregunta.

In [4]:
# TODO: definitions only, no calls here

## 5. Modelos y métricas

TODO: qué modelo o método se usa, contra qué línea base se compara y con qué métrica.

In [5]:
# TODO: definitions only, no calls here

## 6. Gráficas

Funciones para mostrar resultados. `imprimir_tabla` imprime los registros por iteración como tabla de texto (no se usa `pandas`). Las funciones de gráficas guardan la figura en `figuras/` y devuelven `fig, ax`.

In [6]:
ROLE = {"actual": "#0E4D45", "model": "#3A7CA5", "compare": "#B07D2B",
        "up": "#3F8A26", "down": "#A4493D", "missing": "#C9CFCB", "shade": "#EEF0EE"}


def style_ax(ax, title, xlabel, ylabel, subtitle=None):
    """Title, axis labels and clean spines in the house style."""
    ax.set_title(title + (f"\n{subtitle}" if subtitle else ""), fontsize=16, fontweight="bold", pad=18)
    ax.set_xlabel(xlabel, fontsize=13, fontweight="bold", labelpad=12)
    ax.set_ylabel(ylabel, fontsize=13, fontweight="bold", labelpad=12)
    ax.spines[["top", "right"]].set_visible(False)
    ax.tick_params(labelsize=12)
    ax.grid(axis="y", alpha=0.3)


def label_points(ax, x, y, fmt="{:.1f}", every=1, color=None):
    """Bold value labels past the end of every Nth point or bar; negatives go below, in clay."""
    y = list(y)
    span = (max(y) - min(y)) or abs(max(y)) or 1
    for i, (xi, yi) in enumerate(zip(x, y)):
        if i % every == 0:
            below = yi < 0
            ax.text(xi, yi + (-1 if below else 1) * span * 0.03, fmt.format(yi), ha="center",
                    va="top" if below else "bottom", fontsize=10, fontweight="bold",
                    color=ROLE["down"] if below else (color or ROLE["actual"]))


def ref_line(ax, value, label, kind="target"):
    """Target (dashed clay), average (solid ochre) or zero (thin gray) line."""
    styles = {"target": dict(color=ROLE["down"], linestyle="--", linewidth=1.8),
              "average": dict(color=ROLE["compare"], linestyle="-", linewidth=2),
              "zero": dict(color="#444444", linestyle="--", linewidth=0.8)}
    ax.axhline(value, label=label, zorder=2, **styles[kind])


def imprimir_tabla(filas, columnas):
    """Print a list of dicts as a fixed-width text table, one row per dict, in the given column order."""
    texto = [[str(f[c]) for c in columnas] for f in filas]
    anchos = [max([len(c)] + [len(t[i]) for t in texto]) for i, c in enumerate(columnas)]
    print(" | ".join(c.ljust(a) for c, a in zip(columnas, anchos)))
    print("-+-".join("-" * a for a in anchos))
    for t in texto:
        print(" | ".join(v.ljust(a) for v, a in zip(t, anchos)))

## 7. Ejecución

Cada paso llama a las funciones de arriba. Después de cada resultado va su interpretación con los números obtenidos.

### 7.1 Parte 1 | Act. 1: distancia del tour inicial

In [7]:
# TODO: call the functions for this step

TODO: interpretación de este resultado con los números reales.

### 7.2 Parte 1 | Act. 2: vecinos por inversión

In [8]:
# TODO: call the functions for this step

TODO: interpretación de este resultado con los números reales.

### 7.3 Parte 1 | Act. 3 y 4: Hill Climbing de máximo descenso

In [9]:
# TODO: call the functions for this step

TODO: interpretación de este resultado con los números reales.

### 7.4 Parte 1 | Act. 5: 100 reinicios aleatorios

In [10]:
# TODO: call the functions for this step

TODO: interpretación de este resultado con los números reales.

### 7.5 Parte 1 | Extra: fuerza bruta de los tours válidos

In [11]:
# TODO: call the functions for this step

TODO: interpretación de este resultado con los números reales.

### 7.6 Parte 1 | Act. 6: gráfica del mejor recorrido

In [12]:
# TODO: call the functions for this step

TODO: interpretación de este resultado con los números reales.

### 7.7 Parte 2 | Act. 7: heurística euclidiana

In [13]:
# TODO: call the functions for this step

TODO: interpretación de este resultado con los números reales.

### 7.8 Parte 2 | Act. 8 a 10: A\* paso a paso

In [14]:
# TODO: call the functions for this step

TODO: interpretación de este resultado con los números reales.

### 7.9 Parte 2 | Act. 11: A\* con h(n) = 0

In [15]:
# TODO: call the functions for this step

TODO: interpretación de este resultado con los números reales.

### 7.10 Parte 2 | Act. 12: gráfica de la red

In [16]:
# TODO: call the functions for this step

TODO: interpretación de este resultado con los números reales.

### 7.11 Parte 3A | Act. 13: red residual

In [17]:
# TODO: call the functions for this step

TODO: interpretación de este resultado con los números reales.

### 7.12 Parte 3A | Act. 14 a 16: Edmonds-Karp

In [18]:
# TODO: call the functions for this step

TODO: interpretación de este resultado con los números reales.

### 7.13 Parte 3A | Act. 17: verificación de capacidad y conservación

In [19]:
# TODO: call the functions for this step

TODO: interpretación de este resultado con los números reales.

### 7.14 Parte 3A | Extra: corte mínimo

In [20]:
# TODO: call the functions for this step

TODO: interpretación de este resultado con los números reales.

### 7.15 Parte 3B | Act. 18: estado inicial de cada arista

In [21]:
# TODO: call the functions for this step

TODO: interpretación de este resultado con los números reales.

### 7.16 Parte 3B | Act. 19 a 23: costo mínimo paso a paso

In [22]:
# TODO: call the functions for this step

TODO: interpretación de este resultado con los números reales.

### 7.17 Parte 3B | Act. 24: reporte final

In [23]:
# TODO: call the functions for this step

TODO: interpretación de este resultado con los números reales.

### 7.18 Parte 3B | Extra: costo de la distribución de Edmonds-Karp

In [24]:
# TODO: call the functions for this step

TODO: interpretación de este resultado con los números reales.

## 8. Conclusiones

TODO: respuesta a la pregunta con números, limitaciones y siguientes pasos.